In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :memoryless

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [memoryless_model] Fitting chain 3 (tau=59)
[ Info: [memoryless] iter 1000/1000000 elapsed=5.6s, rate=0.183, mean=[0.772, 0.00271, 1.192], std=[0.0386, 0.000469, 0.1660] [ADAPT]
[ Info: [memoryless] iter 2000/1000000 elapsed=10.3s, rate=0.167, mean=[0.754, 0.00225, 1.150], std=[0.0345, 0.000641, 0.1249] [ADAPT]
[ Info: [memoryless] iter 3000/1000000 elapsed=14.3s, rate=0.155, mean=[0.758, 0.00176, 1.108], std=[0.0315, 0.000814, 0.1148] [ADAPT]
[ Info: [memoryless] iter 4000/1000000 elapsed=18.2s, rate=0.147, mean=[0.757, 0.00154, 1.091], std=[0.0279, 0.000784, 0.1034] [ADAPT]
[ Info: [memoryless] iter 5000/1000000 elapsed=22.2s, rate=0.148, mean=[0.763, 0.00138, 1.081], std=[0.0275, 0.000763, 0.0944] [ADAPT]
[ Info: [memoryless] iter 6000/1000000 elapsed=26.2s, rate=0.148, mean=[0.759, 0.00132, 1.071], std=[0.0279, 0.000718, 0.0885] [ADAPT]
[ Info: [memoryless] iter 7000/1000000 elapsed=30.1s, rate=0.146, mean=[0.758, 0.00124, 1.062], std=[0.0262, 0.000688, 0.0844] [ADAPT]
[ In